# 🚲 Bike Sharing Demand — Geliştirilmiş Pipeline
**Kaggle Competition | LightGBM + Optuna + Feature Engineering**

Bu notebook orijinal çözümün geliştirilmiş hâlidir. Yapılan başlıca iyileştirmeler:
- Keşifsel veri analizi (EDA) hücreleri
- Daha zengin özellik mühendisliği (peak-hour, temp_binned, vb.)
- Optuna ile hiperparametre optimizasyonu
- Early stopping + callback desteği
- Model ve encoder kaydetme (HuggingFace deploy için)
- RMSLE'nin yanı sıra MAE / R² metrikleri

In [ ]:
# ── Kütüphaneler ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, joblib
warnings.filterwarnings('ignore')

from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_log_error, mean_absolute_error, r2_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
print('Tüm kütüphaneler yüklendi ✓')

In [ ]:
# ── 1. Veri Yükleme ───────────────────────────────────────────────────────────
# Kaggle ortamında:
DATA_DIR = '/kaggle/input/bike-sharing-demand'

# Yerel ortamda train.csv / test.csv aynı klasördeyse:
if not os.path.exists(DATA_DIR):
    DATA_DIR = '.'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
train.head()

In [ ]:
# ── 2. Keşifsel Veri Analizi ──────────────────────────────────────────────────
print('=== Eksik Değerler ===')
print(train.isnull().sum())

print('\n=== Temel İstatistikler ===')
train.describe()

In [ ]:
# Saatlik ve mevsimsel dağılım
train['datetime'] = pd.to_datetime(train['datetime'])
train['hour']     = train['datetime'].dt.hour

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Saatlik ortalama kiralama
train.groupby('hour')['count'].mean().plot(ax=axes[0], kind='bar', color='steelblue')
axes[0].set_title('Saate Göre Ortalama Kiralama')
axes[0].set_xlabel('Saat')

# Mevsimsel dağılım
season_map = {1:'İlkbahar', 2:'Yaz', 3:'Sonbahar', 4:'Kış'}
train['season_name'] = train['season'].map(season_map)
train.groupby('season_name')['count'].mean().plot(ax=axes[1], kind='bar', color='coral')
axes[1].set_title('Mevsime Göre Ortalama Kiralama')

# Hedef dağılımı
axes[2].hist(np.log1p(train['count']), bins=40, color='mediumseagreen', edgecolor='white')
axes[2].set_title('log1p(count) Dağılımı')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA grafikleri kaydedildi: eda_plots.png')

In [ ]:
# Korelasyon ısı haritası
num_cols = ['temp', 'atemp', 'humidity', 'windspeed', 'count']
corr = train[num_cols].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=.5)
plt.title('Sayısal Değişkenler Korelasyon Matrisi')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Geliştirilmiş Özellik Mühendisliği ────────────────────────────────────
def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])

    # Temel zaman özellikleri
    df['hour']       = df['datetime'].dt.hour
    df['day']        = df['datetime'].dt.day
    df['month']      = df['datetime'].dt.month
    df['dayofweek']  = df['datetime'].dt.dayofweek
    df['year']       = df['datetime'].dt.year
    df['quarter']    = df['datetime'].dt.quarter

    # YENİ: Yoğun saat bayrakları (sabah & akşam rush hour)
    df['is_rush_morning'] = df['hour'].between(7, 9).astype(int)
    df['is_rush_evening'] = df['hour'].between(17, 19).astype(int)

    # YENİ: Gece/gündüz
    df['is_daytime'] = df['hour'].between(6, 22).astype(int)

    # YENİ: Hafta sonu bayrağı
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

    # YENİ: Sıcaklık kümeleme (düşük/orta/yüksek)
    df['temp_bin'] = pd.cut(
        df['temp'],
        bins=[0, 10, 20, 30, 50],
        labels=[0, 1, 2, 3]
    ).astype(int)

    # YENİ: Nem & rüzgar etkileşim özelliği
    df['hum_wind'] = df['humidity'] * df['windspeed']

    # YENİ: Hissedilen sıcaklık farkı
    df['temp_diff'] = df['atemp'] - df['temp']

    # Kategorik dönüşüm
    cat_cols = ['season', 'holiday', 'workingday', 'weather',
                'hour', 'month', 'year', 'dayofweek', 'quarter', 'temp_bin']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')

    return df

train = prepare_data(train)
test  = prepare_data(test)

FEATURE_COLS = [
    'season', 'holiday', 'workingday', 'weather',
    'temp', 'atemp', 'humidity', 'windspeed',
    'hour', 'day', 'month', 'dayofweek', 'year', 'quarter',
    'is_rush_morning', 'is_rush_evening', 'is_daytime',
    'is_weekend', 'temp_bin', 'hum_wind', 'temp_diff'
]

y_log = np.log1p(train['count'])
X     = train[FEATURE_COLS]
X_test = test[FEATURE_COLS]

print(f'Özellik sayısı: {X.shape[1]}')
print(f'Eğitim örneği : {X.shape[0]}')

In [ ]:
# ── 4. Train / Validation Split ───────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y_log, test_size=0.2, random_state=SEED
)
print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}')

In [ ]:
# ── 5. Optuna ile Hiperparametre Optimizasyonu ────────────────────────────────
def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 300, 2000),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves'      : trial.suggest_int('num_leaves', 20, 200),
        'max_depth'       : trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'random_state'    : SEED,
        'n_jobs'          : -1,
        'verbose'         : -1,
    }
    model = LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(50, verbose=False), log_evaluation(-1)]
    )
    preds = np.clip(model.predict(X_val), 0, None)
    return np.sqrt(mean_squared_log_error(np.expm1(y_val), np.expm1(preds)))

study = optuna.create_study(direction='minimize', study_name='bike-lgbm')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n✅ En iyi RMSLE: {study.best_value:.4f}')
print(f'   En iyi parametreler: {study.best_params}')

In [ ]:
# ── 6. Nihai Model Eğitimi (Tüm Veri) ────────────────────────────────────────
best_params = study.best_params
best_params.update({'random_state': SEED, 'n_jobs': -1, 'verbose': -1})

# Önce doğrulama seti ile skor teyidi
final_lgbm = LGBMRegressor(**best_params)
final_lgbm.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[early_stopping(50, verbose=False), log_evaluation(100)]
)

val_preds  = np.clip(final_lgbm.predict(X_val), 0, None)
val_actual = np.expm1(y_val)
val_pred_exp = np.expm1(val_preds)

rmsle = np.sqrt(mean_squared_log_error(val_actual, val_pred_exp))
mae   = mean_absolute_error(val_actual, val_pred_exp)
r2    = r2_score(val_actual, val_pred_exp)

print(f'\n📊 Doğrulama Metrikleri:')
print(f'   RMSLE : {rmsle:.4f}')
print(f'   MAE   : {mae:.2f} bisiklet')
print(f'   R²    : {r2:.4f}')

In [ ]:
# ── 7. Özellik Önem Grafiği ───────────────────────────────────────────────────
importances = pd.Series(
    final_lgbm.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

plt.figure(figsize=(9, 7))
importances.plot(kind='barh', color='steelblue')
plt.title('LightGBM — Özellik Önem Sıralaması', fontsize=13)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: feature_importance.png')

In [ ]:
# ── 8. Tüm Eğitim Verisiyle Yeniden Eğitim ───────────────────────────────────
# (Submission için son model — tüm train verisi kullanılır)
full_model = LGBMRegressor(**best_params)
full_model.fit(X, y_log, callbacks=[log_evaluation(100)])
print('Nihai model tüm veriyle eğitildi ✓')

In [ ]:
# ── 9. Model Kaydetme (HuggingFace için) ──────────────────────────────────────
joblib.dump(full_model, 'bike_model.pkl')
joblib.dump(FEATURE_COLS, 'feature_cols.pkl')
print('bike_model.pkl ve feature_cols.pkl kaydedildi ✓')

In [ ]:
# ── 10. Kaggle Submission ─────────────────────────────────────────────────────
final_preds = np.clip(np.expm1(full_model.predict(X_test)), 0, None)

submission = pd.DataFrame({
    'datetime': test['datetime'],
    'count'   : final_preds.round().astype(int)
})
submission.to_csv('submission.csv', index=False)
print('submission.csv başarıyla oluşturuldu ✓')
submission.head()